# VG_lam=0.05 — Quick Check
**목적**: Validity-Gated λ=0.05 실험  
- SUBSET=5000, SEEDS=[42], BATCH_SIZE=64, EPOCHS=3
- Runtime: T4 GPU 권장


In [ ]:
!pip install kiwipiepy transformers datasets scikit-learn scipy tqdm -q

In [ ]:
import os, json, random, gc
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
from tqdm import tqdm
import warnings; warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
MODEL_NAME   = 'klue/roberta-base'
MAX_LEN      = 128
BATCH_SIZE   = 64
EPOCHS       = 3
LR           = 2e-5
WEIGHT_DECAY = 0.01
LAMBDA       = 0.05
SEEDS        = [42]
SUBSET       = 5000
OUT_DIR      = '/content/vg_lam005_exp'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
SWAP_PAIRS_BY_CAT = {
    'gender':     [('여성', '남성'), ('여자', '남자'), ('여성들', '남성들'),
                   ('여자들', '남자들'), ('페미니스트', '남성우월주의자'),
                   ('페미', '한남'), ('메갈', '한남'), ('한녀', '한남')],
    'religion':   [('무슬림', '기독교인'), ('이슬람', '기독교'),
                   ('무슬림', '천주교인'), ('이슬람교도', '기독교인')],
    'ethnicity':  [('조선족', '한국인'), ('외국인', '내국인'),
                   ('탈북민', '남한사람'), ('베트남인', '한국인'),
                   ('일본인', '한국인'), ('재일교포', '한국인'),
                   ('동남아인', '한국인')],
    'age':        [('노인', '청년'), ('노년층', '청년층'),
                   ('할머니', '젊은여자'), ('할아버지', '젊은남자')],
    'sexuality':  [('동성애자', '이성애자'), ('게이', '이성애자'),
                   ('레즈비언', '이성애자'), ('성소수자', '이성애자'),
                   ('트랜스젠더', '이성애자'), ('퀴어', '이성애자')],
    'disability': [('장애인', '비장애인'), ('정신장애인', '비장애인'),
                   ('지적장애인', '비장애인')],
}
SWAP_MAP = {}
for _cat, _pairs in SWAP_PAIRS_BY_CAT.items():
    for _a, _b in _pairs:
        SWAP_MAP[_a] = (_b, _cat)
SWAP_KEYS = sorted(SWAP_MAP.keys(), key=len, reverse=True)
print(f'Swap terms: {len(SWAP_KEYS)}개')

In [ ]:
from kiwipiepy import Kiwi
kiwi = Kiwi()

def find_swap(text):
    tokens = kiwi.tokenize(text)
    token_forms = [t.form for t in tokens]
    found = [term for term in SWAP_KEYS if term in token_forms]
    if len(set(found)) >= 2:
        return None, None, None
    if found:
        term = found[0]
        counterpart, cat = SWAP_MAP[term]
        return term, counterpart, cat
    return None, None, None

def _has_batchim(char):
    code = ord(char)
    if not (0xAC00 <= code <= 0xD7A3): return False
    return (code - 0xAC00) % 28 != 0

def _ends_with_rieul(char):
    code = ord(char)
    if not (0xAC00 <= code <= 0xD7A3): return False
    return (code - 0xAC00) % 28 == 8

_JOSA_VOWEL = {
    '이': '가', '을': '를', '은': '는', '과': '와', '아': '야',
    '이나': '나', '이랑': '랑', '이든': '든', '이라고': '라고',
    '이라며': '라며', '이라는': '라는', '이라서': '라서',
    '이라도': '라도', '이라면': '라면', '이란': '란', '이야': '야',
}
_JOSA_CONS = {v: k for k, v in _JOSA_VOWEL.items()}
_ALT_JOSA  = set(_JOSA_VOWEL) | set(_JOSA_CONS) | {'으로', '로'}

def _adjust_josa(new_term, josa):
    if not new_term: return josa
    last = new_term[-1]
    if josa in ('으로', '로'):
        return '으로' if (_has_batchim(last) and not _ends_with_rieul(last)) else '로'
    if _has_batchim(last):
        return _JOSA_CONS.get(josa, josa)
    return _JOSA_VOWEL.get(josa, josa)

def make_swap(text, orig_term, new_term):
    tokens = kiwi.tokenize(text)
    subs = []
    for i, t in enumerate(tokens):
        if t.form == orig_term:
            subs.append((t.start, t.start + t.len, new_term))
            if i + 1 < len(tokens):
                nxt = tokens[i + 1]
                if str(nxt.tag).startswith('J') and nxt.form in _ALT_JOSA:
                    fixed = _adjust_josa(new_term, nxt.form)
                    if fixed != nxt.form:
                        subs.append((nxt.start, nxt.start + nxt.len, fixed))
    if not subs: return text
    for start, end, repl in sorted(subs, key=lambda x: x[0], reverse=True):
        text = text[:start] + repl + text[end:]
    return text

print('kiwi + swap functions ready')

In [ ]:
_SEMANTIC_BLACKLIST = {
    'ethnicity': ['입국', '방역', '국경', '불법체류', '이민', '귀화', '출입국',
                  '난민', '밀입국', '추방', '체류'],
    'gender':    ['위안부', '임신', '출산', '생식', '여학생', '남학생', '자궁',
                  '성폭력', '성폭행', '성매매', '강간', '몰카'],
    'religion':  ['부르카', '히잡', '테러', '지하드', '성전', '탈레반', '샤리아'],
    'age':       ['위안부', '전쟁', '일제', '역사적'],
    'sexuality': ['결혼', '입양', '헌혈', '군대', '병역'],
}
_SEMANTIC_BLACKLIST_STRICT = {
    'ethnicity': _SEMANTIC_BLACKLIST['ethnicity'] + ['식민지', '침략', '전쟁범죄', '역사'],
    'gender':    _SEMANTIC_BLACKLIST['gender']    + ['생리', '군대', '병역', '군필'],
    'religion':  _SEMANTIC_BLACKLIST['religion']  + ['이단', '사이비', '교주', '세뇌'],
    'sexuality': _SEMANTIC_BLACKLIST['sexuality'] + ['에이즈', 'HIV', '성전환'],
    'age':       _SEMANTIC_BLACKLIST['age']        + ['60대', '70대', '80대', '90대',
                                                       '고령', '은퇴', '노후', '요양', '치매'],
    'disability': [],
}
_ASYMMETRIC_PAIRS = {
    ('게이', '이성애자'), ('이성애자', '게이'),
    ('레즈비언', '이성애자'), ('이성애자', '레즈비언'),
    ('트랜스젠더', '이성애자'), ('이성애자', '트랜스젠더'),
    ('성소수자', '이성애자'), ('이성애자', '성소수자'),
    ('퀴어', '이성애자'), ('이성애자', '퀴어'),
    ('할머니', '젊은여자'), ('젊은여자', '할머니'),
    ('할아버지', '젊은남자'), ('젊은남자', '할아버지'),
}
_ASYMMETRIC_PAIRS_STRICT = _ASYMMETRIC_PAIRS | {
    ('조선족', '한국인'), ('탈북민', '남한사람'),
    ('베트남인', '한국인'), ('재일교포', '한국인'),
    ('동남아인', '한국인'), ('외국인', '내국인'),
    ('무슬림', '기독교인'), ('무슬림', '천주교인'),
    ('이슬람', '기독교'), ('이슬람교도', '기독교인'),
}
_COMPARISON_TOKENS = {'보다', '처럼', '만큼', '대비', '반면', '달리'}
_EVENT_OBJ_KEYWORDS = {
    '폭행', '살해', '강간', '임신', '출산', '생식', '피해',
    '고소', '처벌', '신고', '학대', '착취',
}

def _check_grammar(cf, swap_term):
    if not swap_term: return True
    last = swap_term[-1]
    if _has_batchim(last):
        bad = [swap_term + j for j in _JOSA_VOWEL.values()]
    else:
        bad = [swap_term + j for j in _JOSA_VOWEL.keys()]
    return not any(p in cf for p in bad)

def compute_validity(original, cf, orig_term, swap_term, cat):
    valid_grammar   = _check_grammar(cf, swap_term)
    blacklist       = _SEMANTIC_BLACKLIST.get(cat, [])
    valid_semantics = not any(kw in original + ' ' + cf for kw in blacklist)
    label_preserving = (
        (orig_term, swap_term) not in _ASYMMETRIC_PAIRS and valid_semantics)
    return {'use_for_ccr': valid_grammar and valid_semantics and label_preserving}

def compute_validity_strict(original, cf, orig_term, swap_term, cat):
    valid_grammar   = _check_grammar(cf, swap_term)
    blacklist       = _SEMANTIC_BLACKLIST_STRICT.get(cat, [])
    valid_semantics = not any(kw in original + ' ' + cf for kw in blacklist)
    label_preserving = (
        (orig_term, swap_term) not in _ASYMMETRIC_PAIRS_STRICT and valid_semantics)
    tokens = kiwi.tokenize(original)
    token_forms = [t.form for t in tokens]
    no_comparison = not any(p in token_forms for p in _COMPARISON_TOKENS)
    no_harmful_obj = True
    for i, t in enumerate(tokens):
        if t.form == orig_term and i + 1 < len(tokens):
            nxt = tokens[i + 1]
            if str(nxt.tag).startswith('J') and nxt.form in ('을', '를'):
                if any(kw in original for kw in _EVENT_OBJ_KEYWORDS):
                    no_harmful_obj = False
                    break
    use_for_ccr = (valid_grammar and valid_semantics and label_preserving
                   and no_comparison and no_harmful_obj)
    return {
        'valid_grammar': valid_grammar, 'valid_semantics': valid_semantics,
        'label_preserving': label_preserving, 'no_comparison': no_comparison,
        'no_harmful_obj': no_harmful_obj, 'use_for_ccr': use_for_ccr,
    }

print('Validity functions ready')

In [ ]:
class HatersDataset(Dataset):
    def __init__(self, examples, tokenizer, max_len, mode='none', cf_lookup=None):
        assert mode in ('none', 'gated')
        self.tok, self.max_len = tokenizer, max_len
        self.items = []
        for text, label, _ in examples:
            cf_text, cf_valid = None, False
            if mode == 'gated':
                if cf_lookup is not None:
                    entry = cf_lookup.get(text)
                    if entry is not None:
                        precomp_cf, orig_term, swap_term, cat = entry
                        cf_text = precomp_cf
                        v = compute_validity(text, cf_text, orig_term, swap_term, cat)
                        cf_valid = v['use_for_ccr']
                else:
                    orig_term, swap_term, cat = find_swap(text)
                    if orig_term is not None:
                        cf_text = make_swap(text, orig_term, swap_term)
                        v = compute_validity(text, cf_text, orig_term, swap_term, cat)
                        cf_valid = v['use_for_ccr']
            self.items.append({'text': text, 'label': label,
                               'cf_text': cf_text, 'cf_valid': cf_valid})

    def _enc(self, text):
        e = self.tok(text, max_length=self.max_len,
                     padding='max_length', truncation=True, return_tensors='pt')
        return e['input_ids'].squeeze(0), e['attention_mask'].squeeze(0)

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        it = self.items[idx]
        ids, mask = self._enc(it['text'])
        cf_src = it['cf_text'] if it['cf_text'] else it['text']
        cf_ids, cf_mask = self._enc(cf_src)
        return {
            'input_ids': ids, 'attention_mask': mask,
            'label': torch.tensor(it['label'], dtype=torch.long),
            'cf_valid': torch.tensor(it['cf_valid'], dtype=torch.bool),
            'cf_input_ids': cf_ids, 'cf_attention_mask': cf_mask,
        }


class HateDetector(nn.Module):
    def __init__(self, model_name, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, 2)

    def forward(self, input_ids, attention_mask):
        cls = self.encoder(input_ids=input_ids,
                           attention_mask=attention_mask).last_hidden_state[:, 0]
        return self.classifier(self.dropout(cls))

    def probs(self, input_ids, attention_mask):
        return F.softmax(self.forward(input_ids, attention_mask), dim=-1)


def sym_kl(p, q):
    p = p.clamp(min=1e-8); q = q.clamp(min=1e-8)
    return (F.kl_div(q.log(), p, reduction='batchmean') +
            F.kl_div(p.log(), q, reduction='batchmean')) / 2


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

print('Dataset / Model classes ready')

In [ ]:
from datasets import load_dataset

HATE_LABELS = {'offensive', 'L1_hate', 'L2_hate'}

def load_khaters(split, subset=0, seed=42):
    ds = load_dataset('humane-lab/K-HATERS', split=split)
    examples = []
    for row in ds:
        text = row['text'].strip()
        if not text: continue
        label   = 1 if row['label'] in HATE_LABELS else 0
        targets = row.get('target_label') or []
        examples.append((text, label, targets))
    if subset:
        rng = random.Random(seed)
        pos = [e for e in examples if e[1] == 1]
        neg = [e for e in examples if e[1] == 0]
        rng.shuffle(pos); rng.shuffle(neg)
        examples = pos[:subset // 2] + neg[:subset // 2]
        rng.shuffle(examples)
    return examples

print('Loading K-HATERS...')
train_data = load_khaters('train',      subset=SUBSET)
val_data   = load_khaters('validation', subset=0)
test_data  = load_khaters('test',       subset=0)
print(f'train={len(train_data)}  val={len(val_data)}  test={len(test_data)}')

In [ ]:
# ── cf_pairs_train.jsonl 로딩 (kiwi find_swap+make_swap 생략) ────────────────
CF_PAIRS_PATH = '/content/cf_pairs_train.jsonl'  # Colab에 업로드한 경로

cf_lookup = None
if os.path.exists(CF_PAIRS_PATH):
    cf_lookup = {}
    with open(CF_PAIRS_PATH, encoding='utf-8') as f:
        for line in f:
            p = json.loads(line)
            cf_lookup[p['original']] = (p['cf'], p['orig_term'], p['swap_term'], p['category'])
    print(f'cf_lookup loaded: {len(cf_lookup)} entries → kiwi 스킵')
else:
    print(f'cf_pairs_train.jsonl 없음 → kiwi fallback 사용')

# gate pass rate 확인 (cf_lookup 기준)
print('
Gate pass rate 확인 중...')
n_total = n_swappable = n_valid = 0
for text, label, _ in tqdm(train_data, desc='Gate check'):
    n_total += 1
    if cf_lookup is not None:
        entry = cf_lookup.get(text)
        if entry is None:
            continue
        n_swappable += 1
        cf_text, orig_term, swap_term, cat = entry
    else:
        orig_term, swap_term, cat = find_swap(text)
        if orig_term is None:
            continue
        n_swappable += 1
        cf_text = make_swap(text, orig_term, swap_term)
    if compute_validity(text, cf_text, orig_term, swap_term, cat)['use_for_ccr']:
        n_valid += 1

print(f'총 샘플       : {n_total}')
print(f'Swappable     : {n_swappable} ({100*n_swappable/max(1,n_total):.1f}%)')
print(f'Standard gate : {n_valid}/{n_swappable}  ({100*n_valid/max(1,n_swappable):.1f}%)')
print(f'CCR 적용 비율 : {100*n_valid/n_total:.1f}% of all train')

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, lam):
    model.train()
    s_total = s_cls = s_cons = 0.0
    for batch in tqdm(loader, desc='  train', leave=False):
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        y     = batch['label'].to(device)
        valid = batch['cf_valid'].to(device)
        optimizer.zero_grad()
        logits   = model(ids, mask)
        cls_loss = F.cross_entropy(logits, y)
        loss     = cls_loss
        c_val    = torch.tensor(0.0, device=device)
        if valid.any():
            cf_ids  = batch['cf_input_ids'].to(device)
            cf_mask = batch['cf_attention_mask'].to(device)
            p_o = model.probs(ids[valid],    mask[valid])
            p_c = model.probs(cf_ids[valid], cf_mask[valid])
            c_val = sym_kl(p_o, p_c)
            loss  = loss + lam * c_val
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        s_total += loss.item(); s_cls += cls_loss.item(); s_cons += c_val.item()
    n = len(loader)
    return s_total / n, s_cls / n, s_cons / n


def eval_f1(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='  eval', leave=False):
            logits = model(batch['input_ids'].to(device),
                           batch['attention_mask'].to(device))
            preds.extend(logits.argmax(-1).cpu().tolist())
            labels.extend(batch['label'].tolist())
    return f1_score(labels, preds, average='macro')


def eval_fairness(model, test_examples, tokenizer):
    model.eval()
    meta = []
    for text, label, _ in test_examples:
        orig_term, swap_term, cat = find_swap(text)
        cf_text = make_swap(text, orig_term, swap_term) if orig_term else None
        meta.append((text, label, cf_text))

    def batch_infer(texts):
        probs_all = []
        for i in range(0, len(texts), BATCH_SIZE):
            bt = texts[i:i + BATCH_SIZE]
            enc = tokenizer(bt, max_length=MAX_LEN, padding='max_length',
                            truncation=True, return_tensors='pt')
            with torch.no_grad():
                logits = model(enc['input_ids'].to(device),
                               enc['attention_mask'].to(device))
            probs_all.extend(F.softmax(logits, dim=-1)[:, 1].cpu().tolist())
        return probs_all

    orig_probs = batch_infer([m[0] for m in meta])
    swap_idx = [i for i, m in enumerate(meta) if m[2] is not None]
    cf_map = {}
    if swap_idx:
        cf_probs = batch_infer([meta[i][2] for i in swap_idx])
        cf_map = dict(zip(swap_idx, cf_probs))

    swap_res = [(orig_probs[i], cf_map[i]) for i in swap_idx]
    flip_rate  = sum(int(p >= 0.5) != int(q >= 0.5) for p, q in swap_res) / max(1, len(swap_res))
    logit_gap  = sum(abs(p - q) for p, q in swap_res) / max(1, len(swap_res))
    return flip_rate, logit_gap


# ── Run ───────────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

va_ds = HatersDataset(val_data, tokenizer, MAX_LEN, mode='none')
va_dl = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

results = {'f1': [], 'flip_rate': [], 'logit_gap': []}

for seed in SEEDS:
    print(f'\n[VG_lam=0.05] seed={seed}')
    set_seed(seed)

    print('  Building train dataset (kiwi)...')
    tr_ds = HatersDataset(train_data, tokenizer, MAX_LEN, mode='gated', cf_lookup=cf_lookup)
    n_cf  = sum(1 for it in tr_ds.items if it['cf_valid'])
    n_sw  = sum(1 for it in tr_ds.items if it['cf_text'] is not None)
    print(f'  Standard gate: {n_cf}/{n_sw} ({100*n_cf/max(1,n_sw):.1f}%)'
          f' = {100*n_cf/len(tr_ds):.1f}% of all train')

    tr_dl = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    model = HateDetector(MODEL_NAME).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps  = len(tr_dl) * EPOCHS
    warmup_steps = max(1, int(0.06 * total_steps))
    scheduler = get_linear_schedule_with_warmup(opt, warmup_steps, total_steps)

    best_f1    = 0.0
    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    for ep in range(1, EPOCHS + 1):
        tl, cl, cons = train_epoch(model, tr_dl, opt, scheduler, LAMBDA)
        vf1 = eval_f1(model, va_dl)
        print(f'  ep{ep}: total={tl:.4f} cls={cl:.4f} cons={cons:.4f} | val_F1={vf1:.4f}')
        if vf1 > best_f1:
            best_f1    = vf1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    te_ds = HatersDataset(test_data, tokenizer, MAX_LEN, mode='none')
    te_dl = DataLoader(te_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_f1 = eval_f1(model, te_dl)
    flip, lgap = eval_fairness(model, test_data, tokenizer)
    print(f'  test F1={test_f1:.4f}  flip={flip:.4f}  logit_gap={lgap:.4f}')

    results['f1'].append(test_f1)
    results['flip_rate'].append(flip)
    results['logit_gap'].append(lgap)

    del model, tr_ds, tr_dl
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
with open(os.path.join(OUT_DIR, 'results.json'), 'w', encoding='utf-8') as f:
    json.dump({'VG_lam=0.05': results}, f, ensure_ascii=False, indent=2)
print(f'Saved to {OUT_DIR}/results.json')

# 기존 결과 비교 (full data 3-seed 기준, 참고용)
prev = {
    'Baseline':       {'f1': 0.7467, 'flip_rate': 0.0319, 'logit_gap': 0.0299},
    'Naive Swap':     {'f1': 0.7454, 'flip_rate': 0.0343, 'logit_gap': 0.0242},
    'Validity-Gated (lam=0.1)': {'f1': 0.7454, 'flip_rate': 0.0343, 'logit_gap': 0.0242},
}

print(f'
{"Model":<28} {"F1":>8} {"Flip":>8} {"LogitGap":>10}  (참고: full-data 3-seed)')
print('-' * 60)
for name, m in prev.items():
    print(f'{name:<28} {m["f1"]:>8.4f} {m["flip_rate"]:>8.4f} {m["logit_gap"]:>10.4f}')
print(f'{"VG_lam=0.05 (now)":<28} '
      f'{np.mean(results["f1"]):>8.4f} '
      f'{np.mean(results["flip_rate"]):>8.4f} '
      f'{np.mean(results["logit_gap"]):>10.4f}')